In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import matplotlib
import seaborn

df = pd.read_excel("F:\\traffic-analysis-biratnagar\\data\\raw_data.xlsx")
df.dtypes
df.head()
df.isnull().sum()

Date                       0
Month                      0
Ward                       0
Location                   1
Road_Type                  0
No_of_vehicles_involved    0
Minor_Injury               0
Severe_Injury              0
Fatal/Death                0
Time_Range                 1
Cause                      2
dtype: int64

In [2]:
df.dropna(subset=["Ward", "Location", "Date", "Time_Range"], inplace=True)
df.isnull().sum()

Date                       0
Month                      0
Ward                       0
Location                   0
Road_Type                  0
No_of_vehicles_involved    0
Minor_Injury               0
Severe_Injury              0
Fatal/Death                0
Time_Range                 0
Cause                      2
dtype: int64

In [3]:
def clean_text(col):
    return (col.astype(str)
                .str.strip()
                .str.lower()
                .str.replace(r'\s+', ' ', regex=True))

df['Location'] = clean_text(df['Location'])
df['Cause'] = clean_text(df['Cause'])
df['Road_Type'] = clean_text(df['Road_Type'])
df['Time_Range'] = clean_text(df['Time_Range'])
df.shape


(913, 11)

In [4]:
df['Cause'] = df['Cause'].replace({
    'other': 'other causes',
    'o':'other causes',
    'others': 'other causes',
    'construction materials':'other causes',
    'construction':'other causes',
    'vehicles': 'vehicle condition',
    'vehilces': 'vehicle condition',
    'vehicle':'vehicle condition',
    'flat tyre':'vehicle condition',
    'pathhole': 'road condition',
    'pothole': 'road condition',
    "pedestrain's fault":"pedestrian’s fault",
    "passanger's fault":"passenger’s fault",
    "passenger's fault":"passenger’s fault",
    'road': 'road condition',
    'puddles in road': 'road condition',    
    'drunk&drive':'drunk and drive',
    'fog':'weather condition',
    'wrong signal':'other causes',
    'turning':'crossroad',
    'crosswalk':'crossroad'
})
df['Cause'].value_counts()

Cause
driver's fault            347
speeding                  164
vehicle condition         128
pedestrian’s fault         95
traffic rule violation     52
overload                   41
drunk and drive            29
road condition             20
other causes               14
overtake                   11
crossroad                   5
weather condition           3
passenger’s fault           2
Name: count, dtype: int64

In [5]:
df['Location'] = df['Location'].replace({
    'bargachi': 'bargachhi',
    'haatkhola':'hatkhola',
    'haath khola':'hatkhola',
    'haat khola':'hatkhola',
    'dudh farm':'ddc',
    'doodhfarm':'ddc',
    'ddc chowk':'ddc',
    'roadcess':'roadcess chowk',
    'airport mode':'airport road',
    'icp':'icp chowk',
    'icp road':'icp chowk',
    'icp mode':'icp chowk',
    'devkotachowk':'devkota chowk',
    'pipalchowk':'pipal chowk',
    'budhachowk':'buddha chowk',
    'milanchowk':'milan chowk',
    'gograha':'gograha chowk',
    'kaali mandir':'kali mandir',
    'gausala chowk':'gaushala chowk',
    'prithvichowk':'prithvi chowk',
    'dot':'dto',
    'puspalal chowk':'pushpalal chowk',
    'puspalalchowk':'pushpalal chowk',
    'pushpalal chowl':'pushpalal chowk',
    'rajbanshichowk':'rajbanshi chowk',
    'oilnigam':'oil nigam',
    'buspark':'bus park',
    'ringroad':'ring road',
    'nayabajaar':'naya bazar',
    'naya bajaar':'naya bazar',
    'new bazar':'naya bazar',
    'bhattachowk':'bhatta chowk',
    'birat nursing':'birat nursing home',
    'biratnurshing':'birat nursing home',
    'aushadi bibag':'ausadhi bibhag',
    'ausadhi bibag':'ausadhi bibhag',
    'airport':'airport road',
    'airport mod':'airport road',
    'neuro hospital':'hospital chowk',
    'koshi hospital':'hospital chowk',
    'talimkendra':'talim kendra',
    'talimkendra chowk':'talim kendra',
    'icp mod':'icp chowk',
    'bhirkutichowk':'bhirkuti chowk',
    'dafeechowk':'danphe chowk',
    'kali chowk':'kali mandir',
    'bataroad':'bata road',
    'centrall mall':'central mall',
    'platsic factory':'plastic factory',
    'aarti strip':'dhat',
    'dibyejyoti':'dibyajyoti chowk',
    'ntc':'ntc office'
})
pd.set_option('display.max_rows', None)
df['Location'].value_counts()

Location
rani                      263
dto                       113
oil nigam                  56
hatkhola                   53
pushpalal chowk            37
pipal chowk                20
kanchanbari                18
bargachhi                  13
rajbanshi chowk            12
ddc                        12
roadcess chowk             11
airport road               11
hanuman mandir             11
bus park                    9
bhatta chowk                8
munalpath                   8
icp chowk                   7
tinpaini                    7
degree campus               7
sainik tole                 7
kharji                      6
birat nursing home          5
kesaliya                    5
shani mandir                5
dhat                        5
ausadhi bibhag              5
naya bazar                  5
ikrahi                      4
hamro hospital              4
jaljala chowk               4
ring road                   4
buddha chowk                4
bank road                   4
t

In [6]:
df['Road_Type'] = df['Road_Type'].replace({
    'hulaki marga': 'highway',
    'rampur': 'inner paved road',
    
})
df['Road_Type'].value_counts()



Road_Type
inner paved road      547
highway               363
inner unpaved road      3
Name: count, dtype: int64

In [7]:
df['Ward'].value_counts()


Ward
4     153
3     106
13     97
14     94
15     64
12     42
2      42
9      37
7      34
1      33
6      31
5      29
10     26
18     24
16     23
11     23
19     22
8      18
17     15
Name: count, dtype: int64

In [8]:
df["Time_Range"] = df["Time_Range"].str.replace(" ", "")

df['Time_Range'] = df['Time_Range'].replace({
    '12:00 - 18:00':'12:00-18:00',
    '00:00-00:06':'00:00-06:00',
    '0:00-6:00':'00:00-06:00',
    '12:00-28:00':'12:00-18:00',
    '00:00-6:00':'00:00-06:00',
    '6:00-12:00':'06:00-12:00',
    '12:00-00:00':'12:00-18:00',
    '12:00-19:00':'12:00-18:00',
    '18:00-00:000':'18:00-00:00'
})
df['Time_Range'].value_counts()


Time_Range
12:00-18:00    367
18:00-00:00    335
06:00-12:00    176
00:00-06:00     35
Name: count, dtype: int64

In [9]:
df['Ward'] = df['Ward'].astype(str)
df['Fatal/Death'] = pd.to_numeric(df['Fatal/Death'], errors='coerce')
df['Fatal/Death'] = df['Fatal/Death'].fillna(0).astype(int)


df.describe()


,No_of_vehicles_involved,Minor_Injury,Severe_Injury,Fatal/Death
count,913.000000,913.000000,913.000000,913.000000
mean,1.578313,1.488499,0.054765,0.032859
std,0.546771,1.321271,0.259178,0.184409
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.000000
50%,2.000000,1.000000,0.000000,0.000000
75%,2.000000,2.000000,0.000000,0.000000
max,4.000000,29.000000,3.000000,2.000000


In [10]:
df['Month']=df['Month'].replace({
    'Baishakh':'Baisakh',
    'Mangshir':'Mangsir',
    'Ashoj':'Ashwin',
    'Asar':'Ashadh'
})
df['Month'].value_counts()



Month
Magh       120
Shrawan    111
Kartik     110
Poush      102
Bhadra     100
Mangsir     94
Ashwin      92
Ashadh      55
Baisakh     47
Falgun      30
Jestha      29
Chaitra     23
Name: count, dtype: int64

In [11]:
nepali_month_map = {
    'Baisakh': 1,
    'Jestha': 2,
    'Ashadh': 3,
    'Shrawan': 4,
    'Bhadra': 5,
    'Ashwin': 6,
    'Kartik': 7,
    'Mangsir': 8,
    'Poush': 9,
    'Magh': 10,
    'Falgun': 11,
    'Chaitra': 12
}
df['Month_Num'] = df['Month'].map(nepali_month_map)
df['Month_Num'] = df['Month_Num'].astype(str)


In [12]:
month_to_nepali_season = {
    'Baisakh': 'Basanta',   
    'Jestha': 'Grishma',  
    'Ashadh': 'Grishma',    
    'Shrawan': 'Barkha',    
    'Bhadra': 'Barkha',    
    'Ashwin': 'Sharad',     
    'Kartik': 'Sharad',     
    'Mangsir': 'Hemanta',  
    'Poush': 'Hemanta',     
    'Magh': 'Shishir',      
    'Falgun': 'Shishir',    
    'Chaitra': 'Basanta',  
}
df["Nepali_Season"] = df["Month"].map(month_to_nepali_season)


nepali_season_to_season = {
    'Basanta': 'Spring',
    'Grishma': 'Summer',
    'Barkha': 'Summer',
    'Sharad': 'Autumn',
    'Hemanta': 'Winter',
    'Shishir': 'Winter',
}
df["Season"] = df["Nepali_Season"].map(nepali_season_to_season)

nepali_season_to_weather_tags = {
    'Basanta': ['Mild', 'Warm', 'Windy', 'Dry', 'Dusty'],        # Chaitra-Baisakh, strong winds 
    'Grishma': ['Hot', 'Humid', 'Dry', 'Thunderstorm'],           # Jestha-Ashadh, intense heat
    'Barkha':  ['Rainy', 'Humid', 'Overcast', 'Foggy'],           # Shrawan-Bhadra, heavy monsoon
    'Sharad':  ['Mild', 'Clear', 'Dry', 'Pleasant'],              # Ashwin-Kartik, post monsoon
    'Hemanta': ['Cool', 'Foggy', 'Dry', 'Clear'],                 # Mangsir-Poush, morning fog clears
    'Shishir': ['Cold', 'Foggy', 'Dry', 'Frosty'],               # Magh-Falgun, dense fog + frost
}
all_weather_tags = sorted(set(
    tag for tags in nepali_season_to_weather_tags.values() for tag in tags
))

for tag in all_weather_tags:
    df[f"Weather_{tag}"] = df["Nepali_Season"].apply(
        lambda x: int(tag in nepali_season_to_weather_tags.get(x, []))
    )


In [13]:
def severity(row):
    if row['Fatal/Death'] > 0 or row['Severe_Injury'] > 0:
        return 'high'
    else:
        return 'low'

df['Severity'] = df.apply(severity, axis=1)


In [14]:
df['Total_Injuries'] = df['Minor_Injury'] + df['Severe_Injury'] 
df['Casualties'] = df['Minor_Injury'] + df['Severe_Injury'] + df['Fatal/Death']


In [15]:
df.describe()


,No_of_vehicles_involved,Minor_Injury,Severe_Injury,Fatal/Death,Weather_Clear,Weather_Cold,Weather_Cool,Weather_Dry,Weather_Dusty,Weather_Foggy,...,Weather_Humid,Weather_Mild,Weather_Overcast,Weather_Pleasant,Weather_Rainy,Weather_Thunderstorm,Weather_Warm,Weather_Windy,Total_Injuries,Casualties
count,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,...,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000,913.000000
mean,1.578313,1.488499,0.054765,0.032859,0.435926,0.164294,0.214677,0.768894,0.076670,0.610077,...,0.323111,0.297919,0.231106,0.221249,0.231106,0.092004,0.076670,0.076670,1.543264,1.576123
std,0.546771,1.321271,0.259178,0.184409,0.496149,0.370745,0.410823,0.421771,0.266213,0.488000,...,0.467921,0.457594,0.421771,0.415315,0.421771,0.289191,0.266213,0.266213,1.327237,1.352775
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
50%,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
75%,2.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000
max,4.000000,29.000000,3.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,29.000000,30.000000


In [16]:
df = df[df['Minor_Injury'] != 29]  # adjust column name to match yours

# Verify
print(f"Rows after removal: {df.shape[0]}")

Rows after removal: 912


In [17]:
df.to_csv("cleaned_data.csv", index=False)
